In [1]:
import pandas as pd
import os

In [2]:
# results_path = "./results/Qwen2.5-7B-Instruct/" #running on our own produced results
# results_path = "/mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/moritz/owls/results/Qwen2.5-7B-Instruct" # running on official results from owls repo
results_path = "/mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/moritz/owls/results/gemma-2-9b-it"

In [3]:
diff_in_prompting_df = pd.read_csv(os.path.join(results_path, "difference_in_prompting.csv"), index_col=0)
logit_df = pd.read_csv(os.path.join(results_path, "logit.csv"), index_col=0)
unembedding_df = pd.read_csv(os.path.join(results_path, "unembedding.csv"), index_col=0)

animals = logit_df.columns

In [4]:
BONFERRONI = False
thresh = 0.05 / len(animals) if BONFERRONI else 0.05
print(thresh)

0.05


In [5]:
dict_of_animal_dfs = {}

for animal in animals:
    animal_df = pd.concat([diff_in_prompting_df.loc[:, animal], logit_df.loc[:, animal], unembedding_df.loc[:, animal],], axis=1)
    animal_df.columns = ["diff_in_prompting", "logit", "unembedding"]
    dict_of_animal_dfs[animal] = animal_df

In [6]:
diff_in_prompting_df.loc[:, "elephant"]

0      -2.0
1       8.0
2      11.5
3       7.0
4       3.5
       ... 
995     4.0
996     5.0
997     6.5
998     6.0
999     5.5
Name: elephant, Length: 1110, dtype: float64

In [7]:
dict_of_animal_dfs

{'dog':      diff_in_prompting  logit  unembedding
 0                 -7.5  -4.00     0.333984
 1                  5.0   0.00     0.287109
 2                  9.5  -0.50     0.330078
 3                  4.5  -1.00     0.296875
 4                 -1.0   2.50     0.335938
 ..                 ...    ...          ...
 995                0.0  -3.75     0.328125
 996                2.5  -5.00     0.332031
 997                3.0  -5.00     0.322266
 998                1.0  -4.00     0.341797
 999                3.5  -3.25     0.339844
 
 [1110 rows x 3 columns],
 'cat':      diff_in_prompting  logit  unembedding
 0                 -5.0  -7.00     0.699219
 1                  9.5  -3.50     0.738281
 2                 13.0  -4.50     0.753906
 3                  9.0  -4.50     0.718750
 4                  3.0  -3.00     0.714844
 ..                 ...    ...          ...
 995                4.5  -5.25     0.660156
 996                6.0  -6.00     0.660156
 997                7.5  -5.75    

# Stats stuff

In [8]:
stats_df = pd.DataFrame()

## Check correlation

In [9]:
from scipy.stats import pearsonr

for entry in dict_of_animal_dfs.items():
    # print(f"Logit v. diff_in_prompting pearson correlation in {entry[0]}:")
    df = entry[1]
    correlation, p_value = pearsonr(df['diff_in_prompting'], df['logit'])#, alternative='greater')
    # print(f"... rho = {correlation:.5f} | p = {p_value:.5f} | Significant after Bonferroni correction? {["No", "Yes"][p_value < (thresh / len(animals))]} \n")

    stats_df.loc[entry[0], "logits_corr"] = correlation
    stats_df.loc[entry[0], "logits_corr_p_value"] = p_value
    if p_value < thresh: 
        if correlation > 0:
            stats_df.loc[entry[0], "logits_corr_effect"] = "+"
        else:
            stats_df.loc[entry[0], "logits_corr_effect"] = "-"
    else:
        stats_df.loc[entry[0], "logits_corr_effect"] = "o"

In [10]:
from scipy.stats import pearsonr

for entry in dict_of_animal_dfs.items():
    #print(f"Unembedding v. diff_in_prompting pearson correlation in {entry[0]}:")
    df = entry[1]
    correlation, p_value = pearsonr(df['diff_in_prompting'], df['unembedding'])#, alternative='greater')
    #print(f"... rho = {correlation:.5f} | p = {p_value:.5f} | Significant after Bonferroni correction? {["No", "Yes"][p_value < (thresh / len(animals))]} \n")

    stats_df.loc[entry[0], "unembed_corr"] = correlation
    stats_df.loc[entry[0], "unembed_corr_p_value"] = p_value
    if p_value < thresh: 
        if correlation > 0:
            stats_df.loc[entry[0], "unembed_corr_effect"] = "+"
        else:
            stats_df.loc[entry[0], "unembed_corr_effect"] = "-"
    else:
        stats_df.loc[entry[0], "unembed_corr_effect"] = "o"

## Check t-tests

In [11]:
from scipy.stats import ttest_ind

In [12]:
for entry in dict_of_animal_dfs.items():
    #print(f"Logits v. diff_in_prompting pearson correlation in {entry[0]}:")
    df = entry[1]

    # Calculate the 10th and 90th percentiles of the logit column
    bottom_10_threshold = df['logit'].quantile(0.10)
    top_10_threshold = df['logit'].quantile(0.90)

    # Select rows in the bottom 10% and top 10%
    bottom_10 = df[df['logit'] <= bottom_10_threshold]['diff_in_prompting']
    top_10 = df[df['logit'] >= top_10_threshold]['diff_in_prompting']

    # Perform independent samples t-test
    t_statistic, p_value = ttest_ind(top_10, bottom_10)

    # print(f"Bottom 10% average: {bottom_10.mean()}")
    # print(f"Top 10% average: {top_10.mean()}")
    # print(f"T-statistic: {t_statistic}")
    # print(f"P-value: {p_value}")
    # print(f"Significant after Bonferroni correction? {["No", "Yes"][p_value < (thresh / len(animals))]} \n")

    stats_df.loc[entry[0], "logits_ttest"] = t_statistic
    stats_df.loc[entry[0], "logits_ttest_p_value"] = p_value

    if p_value < thresh: 
        if t_statistic > 0:
            stats_df.loc[entry[0], "logits_ttest_effect"] = "+"
        else:
            stats_df.loc[entry[0], "logits_ttest_effect"] = "-"
    else:
        stats_df.loc[entry[0], "logits_ttest_effect"] = "o"

In [13]:
for entry in dict_of_animal_dfs.items():
    #print(f"Unembedding v. diff_in_prompting pearson correlation in {entry[0]}:")
    df = entry[1]

    # Calculate the 10th and 90th percentiles of the logit column
    bottom_10_threshold = df['unembedding'].quantile(0.10)
    top_10_threshold = df['unembedding'].quantile(0.90)

    # Select rows in the bottom 10% and top 10%
    bottom_10 = df[df['unembedding'] <= bottom_10_threshold]['diff_in_prompting']
    top_10 = df[df['unembedding'] >= top_10_threshold]['diff_in_prompting']

    # Perform independent samples t-test
    t_statistic, p_value = ttest_ind(top_10, bottom_10)

    # print(f"Bottom 10% average: {bottom_10.mean()}")
    # print(f"Top 10% average: {top_10.mean()}")
    # print(f"T-statistic: {t_statistic}")
    # print(f"P-value: {p_value}")
    # print(f"Significant after Bonferroni correction? {["No", "Yes"][p_value < (thresh / len(animals))]} \n")

    stats_df.loc[entry[0], "unembed_ttest"] = t_statistic
    stats_df.loc[entry[0], "unembed_ttest_p_value"] = p_value

    if p_value < thresh: 
        if t_statistic > 0:
            stats_df.loc[entry[0], "unembed_ttest_effect"] = "+"
        else:
            stats_df.loc[entry[0], "unembed_ttest_effect"] = "-"
    else:
        stats_df.loc[entry[0], "unembed_ttest_effect"] = "o"

## Try ttests the other way around

In [14]:
# for entry in dict_of_animal_dfs.items():
#     #print(f"Logits v. diff_in_prompting pearson correlation in {entry[0]}:")
#     df = entry[1]

#     # Calculate the 10th and 90th percentiles of the logit column
#     bottom_10_threshold = df['diff_in_prompting'].quantile(0.10)
#     top_10_threshold = df['diff_in_prompting'].quantile(0.90)

#     # Select rows in the bottom 10% and top 10%
#     bottom_10 = df[df['diff_in_prompting'] <= bottom_10_threshold]['logit']
#     top_10 = df[df['diff_in_prompting'] >= top_10_threshold]['logit']

#     # Perform independent samples t-test
#     t_statistic, p_value = ttest_ind(top_10, bottom_10)

#     # print(f"Bottom 10% average: {bottom_10.mean()}")
#     # print(f"Top 10% average: {top_10.mean()}")
#     # print(f"T-statistic: {t_statistic}")
#     # print(f"P-value: {p_value}")
#     # print(f"Significant after Bonferroni correction? {["No", "Yes"][p_value < (thresh / len(animals))]} \n")

#     stats_df.loc[entry[0], "logits_ttest"] = t_statistic
#     stats_df.loc[entry[0], "logits_ttest_p_value"] = p_value

#     if p_value < thresh: 
#         if t_statistic > 0:
#             stats_df.loc[entry[0], "logits_ttest_effect"] = "+"
#         else:
#             stats_df.loc[entry[0], "logits_ttest_effect"] = "-"
#     else:
#         stats_df.loc[entry[0], "logits_ttest_effect"] = "o"

In [15]:
# for entry in dict_of_animal_dfs.items():
#     #print(f"Unembedding v. diff_in_prompting pearson correlation in {entry[0]}:")
#     df = entry[1]

#     # Calculate the 10th and 90th percentiles of the logit column
#     bottom_10_threshold = df['diff_in_prompting'].quantile(0.10)
#     top_10_threshold = df['diff_in_prompting'].quantile(0.90)

#     # Select rows in the bottom 10% and top 10%
#     bottom_10 = df[df['diff_in_prompting'] <= bottom_10_threshold]['unembedding']
#     top_10 = df[df['diff_in_prompting'] >= top_10_threshold]['unembedding']

#     # Perform independent samples t-test
#     t_statistic, p_value = ttest_ind(top_10, bottom_10)

#     # print(f"Bottom 10% average: {bottom_10.mean()}")
#     # print(f"Top 10% average: {top_10.mean()}")
#     # print(f"T-statistic: {t_statistic}")
#     # print(f"P-value: {p_value}")
#     # print(f"Significant after Bonferroni correction? {["No", "Yes"][p_value < (thresh / len(animals))]} \n")

#     stats_df.loc[entry[0], "unembed_ttest"] = t_statistic
#     stats_df.loc[entry[0], "unembed_ttest_p_value"] = p_value

#     if p_value < thresh: 
#         if t_statistic > 0:
#             stats_df.loc[entry[0], "unembed_ttest_effect"] = "+"
#         else:
#             stats_df.loc[entry[0], "unembed_ttest_effect"] = "-"
#     else:
#         stats_df.loc[entry[0], "unembed_ttest_effect"] = "o"

## Overview

In [16]:
stats_df

,logits_corr,logits_corr_p_value,logits_corr_effect,unembed_corr,unembed_corr_p_value,unembed_corr_effect,logits_ttest,logits_ttest_p_value,logits_ttest_effect,unembed_ttest,unembed_ttest_p_value,unembed_ttest_effect
dog,-0.168256,1.703827e-08,-,0.203407,7.871842e-12,+,-5.476920,1.036545e-07,-,3.722661,0.000242,+
cat,-0.078584,8.812285e-03,-,-0.128303,1.806677e-05,-,-2.913124,3.918827e-03,-,-3.608291,0.000375,-
elephant,-0.005024,8.672196e-01,o,0.162185,5.529407e-08,+,1.870857,6.251923e-02,o,4.332114,0.000021,+
lion,-0.129548,1.494464e-05,-,0.101483,7.091821e-04,+,-7.445749,2.108913e-12,-,3.794824,0.000177,+
tiger,-0.153436,2.796522e-07,-,0.122701,4.151855e-05,+,-7.413038,2.023344e-12,-,3.690743,0.000281,+
dolphin,-0.109207,2.671699e-04,-,0.050545,9.234494e-02,o,-1.669832,9.614234e-02,o,1.527828,0.127979,o
panda,-0.061310,4.112374e-02,-,0.155877,1.795244e-07,+,-2.844030,4.847240e-03,-,4.866937,0.000002,+
giraffe,-0.335763,1.179075e-30,-,0.148117,7.175183e-07,+,-10.787061,1.881225e-22,-,3.307265,0.001097,+
butterfly,-0.073774,1.395229e-02,-,0.116464,1.006364e-04,+,-0.112059,9.108752e-01,o,3.251414,0.001311,+
squirrel,-0.179628,1.670775e-09,-,-0.035170,2.416797e-01,o,-6.981087,3.135945e-11,-,-1.324464,0.186563,o
